In [ ]:
#5B. Object detection using Transfer Learning of CNN architectures for the Intel Image 
#Classification Dataset using the below steps: 
#a. Set up the environment 
#b. Loading and preprocessing data 
#c. Choose the pre-trained Model (Transfer Learning)  
#d. Freeze the base layers and add custom layers for fine tuning 
#e. Object detection training 
#f. Compute loss 
#g. Evaluate the Model 
#h. Visualize the prediction  
#i. Fine Tune the Model: Fine-tuning involves unfreezing the top layers of the pre
trained model and retraining it on the new dataset to adapt it better to your task.


# Step a: Set up the environment
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


#-------------------------------------------------------------------
# Step b: Loading and preprocessing data
dataset_path = "../LP IV Updated Datasets/Object Detection/Intel Image Classification"

classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(dataset_path + "/" + d)])
print("Classes found:", classes)

IMG_SIZE = 128
images = []
labels = []

# Load 40 images per category for simple and fast training
for idx, class_name in enumerate(classes):
    folder = dataset_path + "/" + class_name
    files = [f for f in os.listdir(folder) if f.endswith('.jpg')][:40]
    for file in files:
        img_path = folder + "/" + file
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        images.append(np.array(img))
        labels.append(idx)

# Preprocess image pixels for VGG16
X = preprocess_input(np.array(images, dtype='float32'))
y = to_categorical(np.array(labels), len(classes))

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

#---------------------------------------------------------------
# Step c & d: Choose pre-trained Model, Freeze base layers and add custom layers
weights_path = "../LP IV Updated Datasets/Object Detection/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
weights = weights_path if os.path.exists(weights_path) else 'imagenet'

# Step c: Choose pre-trained VGG16 model
base_model = VGG16(weights=weights, include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Step d: Freeze base layers
base_model.trainable = False

# Add custom classification layers
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(64, activation='relu'))
model.add(Dense(len(classes), activation='softmax'))

model.summary()


#--------------------------------------------------------------
# Step e & f: Object detection training and Compute loss
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=5, batch_size=16, validation_split=0.1)

# Step f: Plot training loss and accuracy
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Accuracy')
plt.title('Training Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss', color='red')
plt.title('Training Loss')
plt.show()


#----------------------------------------------------------------------
# Step g & h: Evaluate the Model and Visualize the prediction
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

# Visualize sample predictions
predictions = model.predict(X_test)
plt.figure(figsize=(10, 3))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    disp_img = (X_test[i] - X_test[i].min()) / (X_test[i].max() - X_test[i].min())
    plt.imshow(disp_img)
    true_label = classes[np.argmax(y_test[i])]
    pred_label = classes[np.argmax(predictions[i])]
    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis('off')
plt.show()


#------------------------------------------------------------------
# Step i: Fine Tune the Model
# Unfreeze base layers
base_model.trainable = True

# Recompile with small learning rate
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# Fine-tune for 3 epochs
model.fit(X_train, y_train, epochs=3, batch_size=16)

ft_loss, ft_accuracy = model.evaluate(X_test, y_test)
print("Fine-Tuned Test Accuracy:", ft_accuracy)